# Association Rule Mining (TOR) Pipeline — Journalistic News Articles

This notebook executes the end-to-end NLP and semantic association pipeline for news datasets:
1. **Preprocessing**: text cleaning, segmentation, and entity/pattern tokenization using SpaCy.
2. **Trie of Rules (TOR) Construction**: frequent itemset mining and direct **vector PDF** graph visualization.
3. **Cluster Decomposition**: partitioning into central core vs. peripheral sub-clusters.
4. **Semantic Analytics**: Key-Word-In-Context (KWIC) concordance and rule-density-based article recommendation.
5. **Auxiliary Visualizations**: word frequency rankings, frequency bar plots, and co-occurrence graphs (`.gdf`).

In [ ]:
import os
import networkx as nx
import utils.pre_processing_news as ppn
import utils.analize_tor as ator
import utils.processing_file as pf
from utils.pipeline import (
    garantir_diretorios,
    gerar_tor,
    separar_e_salvar_clusters,
    salvar_visualizacoes_comuns
)

# Initialize standard output directories
garantir_diretorios()

## 1. Parameters & Configuration

In [ ]:
# Dataset base filename in datasets/ (without .csv extension)
FILE_NAME = "manifestacao_bolsonaro"
# FILE_NAME = "resgate_juliana_marins"
# FILE_NAME = "morte_juliana_marins"

# Association Rule Mining Algorithm: 'FP-max', 'Apriori', or 'FP-growth'
ALG = "FP-max"
MIN_SUPPORT = 0.05

SEPARAR_PARAGRAFOS = True
IS_TITULO = False

INPUT_FILE = os.path.join("datasets", f"{FILE_NAME}.csv")
COLUMN = "Título" if IS_TITULO else "Texto"

OBS = "p" if SEPARAR_PARAGRAFOS else "n"
OBS = "titulos" if IS_TITULO else OBS
OUTPUT_IDENTIFIER = f"{FILE_NAME}_{OBS}_{ALG}_{MIN_SUPPORT}"
OUTPUT_GRAPHML = os.path.join("outputs", f"{OUTPUT_IDENTIFIER}.graphml")
OUTPUT_PDF = os.path.join("outputs", f"{OUTPUT_IDENTIFIER}_grafo.pdf")

## 2. Preprocessing & Trie of Rules (TOR) Generation

In [ ]:
print(f"Preprocessing database: {INPUT_FILE}...")
pre_processing_result, noticias = ppn.pre_processing_database(
    file_path=INPUT_FILE,
    separar_paragrafos=SEPARAR_PARAGRAFOS,
    column=COLUMN
)
print(f"Total processed transactions/records: {len(pre_processing_result)}")

# Build TOR and save .graphml file
tor = gerar_tor(pre_processing_result, alg=ALG, min_support=MIN_SUPPORT, output_graphml=OUTPUT_GRAPHML)

# Render and export ForceAtlas2 graph to PDF and display inline
G = nx.read_graphml(OUTPUT_GRAPHML)
pf.plotar_grafo(G, output_pdf=OUTPUT_PDF, show=True)

## 3. Cluster Partitioning (Central Core vs. Peripherals)

In [ ]:
G_central, G_perifericos, path_central, path_perifericos = separar_e_salvar_clusters(
    graphml_path=OUTPUT_GRAPHML,
    file_name=OUTPUT_IDENTIFIER
)

## 4. Semantic Analytics: KWIC Concordance & Recommended Article

In [ ]:
# Key-Word-In-Context (KWIC) search
search_term = "20 minutos"
print(f"--- Concordance for keyword: '{search_term}' ---")
ator.concordance(noticias, search_term)

# Recommend article with highest rule density and informativeness
print("\n--- Recommended Article ---")
summary_article = ator.recomendar_texto(noticias, OUTPUT_GRAPHML)
print("\nRecommended Text:\n", summary_article)

## 5. Auxiliary Visualizations

In [ ]:
# Simplified preprocessing for word rankings and co-occurrence
pre_processing_result_simples, _ = ppn.pre_processing_database(
    file_path=INPUT_FILE,
    separar_paragrafos=SEPARAR_PARAGRAFOS,
    column=COLUMN,
    simples=True
)

# Export word ranking CSV, co-occurrence .gdf graph, and plot frequency distribution
df_ranking, file_ranking, file_grafo = salvar_visualizacoes_comuns(
    dados_simples=pre_processing_result_simples,
    file_name=OUTPUT_IDENTIFIER,
    pasta_ranking="ranking_palavras",
    plotar_frequencia=True
)

# Continuous text sample for Word Tree
print("\nContinuous text preview for Word Tree (first 500 characters):")
print(" ".join(noticias)[:500] + "...")